## ML based integration pipeline (Standard Blocker)

In [14]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet, load_csv

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [ ]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators_m2a = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1)
]

comparators_m2g = comparators_m2a + [
    StringComparator(column='genres', similarity_function='jaccard', preprocess=str.lower, list_strategy='concatenate'),
    StringComparator(column='genres', similarity_function='jaccard', preprocess=str.lower, list_strategy='best_match'),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)

train_features_m2a = feature_extractor_m2a.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor_m2g.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [ ]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import MLBasedMatcher

blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['title', 'author'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['title', 'author'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

ml_matcher_m2a = MLBasedMatcher(feature_extractor_m2a)
ml_matcher_m2g = MLBasedMatcher(feature_extractor_m2g)

correspondences_m2a = ml_matcher_m2a.match(
    metabooks, amazon_books,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher_m2g.match(
    metabooks, goodreads,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [9]:
# correspondences_m2a.to_csv(CORR_DIR / "ml_corr_m2a.csv", index=False)
# correspondences_m2g.to_csv(CORR_DIR / "ml_corr_m2g.csv", index=False)

In [10]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

15491

In [ ]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [12]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_standard_blocker.jsonl")

fused_ml_stblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_stblocker):,}')
display(fused_ml_stblocker.head())

Fused rows: 15,115


,_id,_fusion_group_id,_fusion_sources,publisher,publish_year,title,language,genres,price,rating,...,id,isbn_clean,author_norm,author,rating_number,title_norm,_fusion_confidence,_fusion_metadata,edition,book_format
0,metabooks_127120,group_0,"[metabooks, amazon_books]",Villard Books,1985.0,Rush: A Girl's Guide to Sorority Success,English,None,49.990002,5.0,...,metabooks_127120,0394741323,margaret ann rose,Margaret Ann Rose,1,rush a girls guide to sorority success,0.692308,"{'publisher_rule': 'prefer_higher_trust', 'pub...",NaN,NaN
1,amazon_171008,group_1,"[metabooks, amazon_books]",Prentice Hall PTR,1996.0,Bronte Transformaitons: The Cultural Dissemina...,English,"[['Literature', 'Fiction', 'Classics']]",21.250000,5.0,...,amazon_171008,0133555615,patsy stoneman,Patsy Stoneman,3,bronte transformaitons the cultural disseminat...,0.769231,"{'publisher_rule': 'prefer_higher_trust', 'pub...",NaN,NaN
2,metabooks_111040,group_2,"[metabooks, amazon_books]",Bantam Books,1996.0,Potter's Fields,English,"[['Literature', 'Fiction', 'Genre Fiction']]",8.860000,3.8,...,metabooks_111040,055329265X,frank roderus,Frank Roderus,48,potters fields,0.769231,"{'publisher_rule': 'prefer_higher_trust', 'pub...",NaN,NaN
3,amazon_252240,group_3,"[metabooks, amazon_books]",St. Martin's Press,1996.0,The Fire in Autumn,English,"[['Romance', 'Historical']]",7.840000,5.0,...,amazon_252240,0312956908,delia parr,Delia Parr,1,the fire in autumn,0.769231,"{'publisher_rule': 'prefer_higher_trust', 'pub...",NaN,NaN
4,amazon_51573,group_4,"[metabooks, amazon_books]",Macmillan of Canada,1978.0,The eagle and the raven: A novel,English,"[['Literature', 'Fiction', 'World Literature']]",13.660000,4.5,...,amazon_51573,0770517145,pauline gedge,Pauline Gedge,290,the eagle and the raven a novel,0.769231,"{'publisher_rule': 'prefer_higher_trust', 'pub...",NaN,NaN


In [15]:
fused_ml_stblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15115 entries, 0 to 15114
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 15115 non-null  object 
 1   _fusion_group_id    15115 non-null  object 
 2   _fusion_sources     15115 non-null  object 
 3   publisher           15115 non-null  object 
 4   publish_year        15113 non-null  float64
 5   title               15115 non-null  object 
 6   language            15045 non-null  object 
 7   genres              14356 non-null  object 
 8   price               13883 non-null  float64
 9   rating              15115 non-null  float64
 10  page_count          13483 non-null  float64
 11  id                  15115 non-null  object 
 12  isbn_clean          15115 non-null  object 
 13  author_norm         15115 non-null  object 
 14  author              15115 non-null  object 
 15  rating_number       15115 non-null  int64  
 16  titl

In [16]:
fused_ml_stblocker.to_parquet(PIPELINE_DIR / "fused_ml_stblocker.parquet")